In [1]:
import glob
import os 
import copy
import sys
import itertools
import random
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import Lasso, LassoCV
from sklearn.preprocessing import StandardScaler

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchvision import models
from torchvision.models import resnet18, ResNet18_Weights

In [21]:
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger

In [4]:
path = "./"
# The original hard-coded an absolute path here:
#     path = "/Users/amamotoyoshifumi/Desktop/Contrastive/analysis"
# which only exists on the author's machine. Point `path` at the repository root
# (or at the absolute path of your local copy of the dataset).


In [25]:
# The original was:
#     args = sys.argv
#     seed = int(args[1])
# which raises IndexError when the notebook is run interactively.
seed = 0        # controls the group-wise train/test split and the training run


In [36]:
# Label used for the *multimodal* checkpoints (model/Analysis2/<model>/Multi_*).
# NOTE: the single-modality checkpoints loaded in section 4 are hard-coded to PG1.
model = "PG3"


In [27]:
latent_list=[2,4,8,16,32,64,128,256,512]

# 1. Loading Data

In [5]:
df_param=pd.read_csv(path+'/data/params/param_sum.csv', index_col=0)
df_param

,DSC_Tg,"DSC_deltaH,cryst","DSC_deltaH,melt",IR1_mu1,IR1_sigma1,IR1_Int1,IR1_mu2,IR1_sigma2,IR1_Int2,IR2_mu1,...,WAXS_Int10,WAXS_mu11,WAXS_sigma11,WAXS_Int11,Process_Temp,Process_Time,Process_NucConc,Property_Deg,Property_StrainBreak,Property_Young
17_1,60.466362,-47.596012,46.480912,925.000000,8.000000,2.763751e-01,955.131085,6.280011,7.863692,640.601782,...,2660.815097,22.709726,1.452573,106.494511,75,5,0.000,0.078653,0.159800,1533.741152
18_1,61.271702,-48.692377,49.326462,925.000000,8.000000,1.026984e-02,955.259577,5.986762,7.722477,640.861463,...,2290.375137,22.817104,1.521605,94.686515,75,5,0.167,0.086981,0.088500,1595.907404
19_1,62.323349,-15.013878,51.449458,924.157782,4.768348,2.492578e+00,955.615411,5.403198,5.721598,641.002109,...,1377.228359,24.895944,5.824055,1981.239121,75,5,0.500,0.081722,0.057200,1704.734993
21_1,60.937294,-46.231835,44.462586,924.999997,0.000508,1.373156e-10,955.284399,5.796023,7.480258,640.820021,...,2342.947158,24.613239,6.064391,2356.525132,75,10,0.000,0.078401,0.079833,1549.705975
22_1,61.153538,-43.927379,50.978043,925.000000,8.000000,3.040490e-12,955.141682,6.047396,7.448746,640.502364,...,1933.092536,24.754999,5.892152,2031.016989,75,10,0.167,0.087794,0.177067,1499.181431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78_3,59.287018,-1.101862,45.724219,922.401149,5.598432,6.180660e+00,956.429912,5.037394,5.100007,640.000000,...,1042.660663,23.871538,4.654210,1827.687507,120,40,0.167,0.009879,0.034867,1853.110523
79_3,57.929401,-4.225821,42.851180,922.909254,5.136471,5.767580e+00,956.330586,5.145190,4.492747,642.000000,...,612.843045,22.823962,1.414205,133.393312,120,40,0.500,0.010791,0.039867,1812.193412
81_3,61.119339,-41.676676,47.124293,925.000000,8.000000,2.036504e-11,955.205546,6.240761,8.285110,640.686571,...,1978.966766,25.000000,5.593697,2060.534350,0,0,0.000,0.080662,0.116667,1587.940605
82_3,60.671047,-38.773534,48.770991,924.876810,7.999546,8.625324e-11,955.241218,5.947278,7.917637,640.750933,...,2269.634558,24.983987,5.620694,2226.831391,0,0,0.167,0.078363,0.099367,1442.482979


In [6]:
df_dc=df_param[['WAXS_Int1','WAXS_Int2','WAXS_Int3','WAXS_Int4','WAXS_Int5','WAXS_Int6','WAXS_Int7','WAXS_Int8','WAXS_Int9']].sum(axis=1)/df_param[['WAXS_Int1','WAXS_Int2','WAXS_Int3','WAXS_Int4','WAXS_Int5','WAXS_Int6','WAXS_Int7','WAXS_Int8','WAXS_Int9','WAXS_Int10','WAXS_Int11']].sum(axis=1)
df_dc

17_1    0.042973
18_1    0.049297
19_1    0.128264
21_1    0.022538
22_1    0.024244
          ...   
78_3    0.261020
79_3    0.595242
81_3    0.025614
82_3    0.024128
83_3    0.024049
Length: 153, dtype: float64

In [7]:
df_param=pd.read_csv(path+'/data/params/param_sum.csv',index_col=0)
df_pred=pd.concat([df_param[['Property_Deg', 'Property_StrainBreak', 'Property_Young']], df_dc], axis=1)
df_pred['Property_Deg']=df_pred['Property_Deg']*100
df_pred.columns=["Deg","Strain","Young","CrysDeg"]
df_pred["CrysDeg"]=df_pred["CrysDeg"]*100
df_pred

,Deg,Strain,Young,CrysDeg
17_1,7.8653,0.159800,1533.741152,4.297316
18_1,8.6981,0.088500,1595.907404,4.929744
19_1,8.1722,0.057200,1704.734993,12.826432
21_1,7.8401,0.079833,1549.705975,2.253836
22_1,8.7794,0.177067,1499.181431,2.424432
...,...,...,...,...
78_3,0.9879,0.034867,1853.110523,26.101993
79_3,1.0791,0.039867,1812.193412,59.524230
81_3,8.0662,0.116667,1587.940605,2.561380
82_3,7.8363,0.099367,1442.482979,2.412784


In [8]:
df_pred=df_pred.round({
    'Deg': 3,  
    'Strain': 4,  
    'Young': 0,  
    'CrysDeg': 2,  
})
df_pred

,Deg,Strain,Young,CrysDeg
17_1,7.865,0.1598,1534.0,4.30
18_1,8.698,0.0885,1596.0,4.93
19_1,8.172,0.0572,1705.0,12.83
21_1,7.840,0.0798,1550.0,2.25
22_1,8.779,0.1771,1499.0,2.42
...,...,...,...,...
78_3,0.988,0.0349,1853.0,26.10
79_3,1.079,0.0399,1812.0,59.52
81_3,8.066,0.1167,1588.0,2.56
82_3,7.836,0.0994,1442.0,2.41


In [9]:
df_IR=pd.read_csv(path+'/data/measurement/IR.csv', index_col=0)
df_WAXS=pd.read_csv(path+"/data/measurement/WAXS.csv", index_col=0)
POM_list=np.load(path+'/data/measurement/POM2.npy')

In [10]:
df_IR_norm=df_IR.div(df_IR.sum(axis=1), axis=0)
df_WAXS_norm=df_WAXS.div(df_WAXS.sum(axis=1), axis=0)
POM_list_norm=POM_list/np.max(POM_list)
df_IR.shape, df_WAXS.shape, POM_list.shape

((153, 2100), (153, 1000), (153, 496, 880))

In [35]:
X_IR = df_IR_norm.values
y = df_pred.values
print (X_IR.shape, y.shape)

train_groups, test_groups = train_test_split([i for i in range(51)], test_size=0.2, random_state=seed)
print (test_groups)

train_index=train_groups+[i+51 for i in train_groups]+[i+102 for i in train_groups]
test_index=test_groups+[i+51 for i in test_groups]+[i+102 for i in test_groups]
print (test_index)

X_IR_train, y_train=X_IR[train_index].astype("float32"), y[train_index].astype("float32")
X_IR_test, y_test=X_IR[test_index].astype("float32"), y[test_index].astype("float32")

y_train_mean, y_train_std=np.mean(y_train, axis=0), np.std(y_train, axis=0)
y_train=(y_train-y_train_mean)/y_train_std
y_test=(y_test-y_train_mean)/y_train_std

X_IR_train.shape, y_train.shape, X_IR_test.shape, y_test.shape

(153, 2100) (153, 4)
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13]
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13, 94, 91, 97, 63, 75, 82, 68, 83, 54, 81, 64, 145, 142, 148, 114, 126, 133, 119, 134, 105, 132, 115]


((120, 2100), (120, 4), (33, 2100), (33, 4))

In [48]:
X_WAXS = df_WAXS_norm.values
y = df_pred.values
print (X_IR.shape, y.shape)

train_groups, test_groups = train_test_split([i for i in range(51)], test_size=0.2, random_state=seed)
print (test_groups)

train_index=train_groups+[i+51 for i in train_groups]+[i+102 for i in train_groups]
test_index=test_groups+[i+51 for i in test_groups]+[i+102 for i in test_groups]
print (test_index)

X_WAXS_train, y_train=X_WAXS[train_index].astype("float32"), y[train_index].astype("float32")
X_WAXS_test, y_test=X_WAXS[test_index].astype("float32"), y[test_index].astype("float32")

y_train_mean, y_train_std=np.mean(y_train, axis=0), np.std(y_train, axis=0)
y_train=(y_train-y_train_mean)/y_train_std
y_test=(y_test-y_train_mean)/y_train_std

X_WAXS_train.shape, y_train.shape, X_WAXS_test.shape, y_test.shape

(153, 1000) (153, 4)
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13]
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13, 94, 91, 97, 63, 75, 82, 68, 83, 54, 81, 64, 145, 142, 148, 114, 126, 133, 119, 134, 105, 132, 115]


((120, 1000), (120, 4), (33, 1000), (33, 4))

In [52]:
X_POM = POM_list_norm
y = df_pred.values
print (X_POM.shape, y.shape)

train_groups, test_groups = train_test_split([i for i in range(51)], test_size=0.2, random_state=seed)
print (test_groups)

train_index=train_groups+[i+51 for i in train_groups]+[i+102 for i in train_groups]
test_index=test_groups+[i+51 for i in test_groups]+[i+102 for i in test_groups]
print (test_index)

X_POM_train, y_train=X_POM[train_index].astype("float32"), y[train_index].astype("float32")
X_POM_test, y_test=X_POM[test_index].astype("float32"), y[test_index].astype("float32")

y_train_mean, y_train_std=np.mean(y_train, axis=0), np.std(y_train, axis=0)
y_train=(y_train-y_train_mean)/y_train_std
y_test=(y_test-y_train_mean)/y_train_std

X_POM_train.shape, y_train.shape, X_POM_test.shape, y_test.shape

(153, 496, 880) (153, 4)
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13]
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13, 94, 91, 97, 63, 75, 82, 68, 83, 54, 81, 64, 145, 142, 148, 114, 126, 133, 119, 134, 105, 132, 115]


((120, 496, 880), (120, 4), (33, 496, 880), (33, 4))

# 2. IR, WAXS, POM

In [41]:
input_length_IR=X_IR_train.shape[1]
# =========================
# 1. Dataset
# =========================
class IRSpectrumDataset(Dataset):
    def __init__(self, X, y):
        # (N, length) → (N, 1, length)
        X = X[:, None, :]
        
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 4)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class IRModel(pl.LightningModule):
    def __init__(self, input_length, lr=1e-3, latent=8):
        super().__init__()
        self.save_hyperparameters()

        self.model = nn.Sequential(
            # 1st Conv block
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
        
            # 2nd Conv block
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        
            nn.MaxPool1d(kernel_size=2),
        
            # 3rd Conv block
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            # channel compression
            nn.Conv1d(128, 16, kernel_size=1),
            nn.ReLU(),
        
            # flatten
            nn.Flatten(),
        
            # fully connected
            nn.Linear(16 * (input_length // 2), latent),
            nn.ReLU(),
            nn.Dropout(0.2),
        
            nn.Linear(latent, 4)
        )

        self.loss_fn = nn.MSELoss()

        self.train_losses = []
        self.val_losses = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)

        self.train_losses.append(loss.item())
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)

        self.val_losses.append(loss.item())
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

def compute_saliency(model, x):

    model.eval()

    if x.dim() == 3:
        x = x[0]

    x = x.clone().detach()
    x.requires_grad_(True)

    y = model(x.unsqueeze(0)) 

    y = y.squeeze()

    model.zero_grad()
    y.backward()

    saliency = x.grad.abs().squeeze().cpu().numpy()

    return saliency, y.item()

In [49]:
input_length_WAXS=X_WAXS_train.shape[1]
# =========================
# 1. Dataset
# =========================
class WAXSDataset(Dataset):
    def __init__(self, X, y):
        # (N, length) → (N, 1, length)
        X = X[:, None, :]
        
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 4)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# =========================
# 2. Lightning Model
# =========================
class WAXSModel(pl.LightningModule):
    def __init__(self, input_length, lr=1e-3, latent=8):
        super().__init__()
        self.save_hyperparameters()

        self.model = nn.Sequential(
            # 1st Conv block
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
        
            # 2nd Conv block
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        
            nn.MaxPool1d(kernel_size=2),
        
            # 3rd Conv block
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            # channel compression
            nn.Conv1d(128, 16, kernel_size=1),
            nn.ReLU(),
        
            # flatten
            nn.Flatten(),
        
            # fully connected
            nn.Linear(16 * (input_length // 2), latent),
            nn.ReLU(),
            nn.Dropout(0.2),
        
            nn.Linear(latent, 4)
        )

        self.loss_fn = nn.MSELoss()
        self.train_losses = []
        self.val_losses = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)

        self.train_losses.append(loss.item())
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)

        self.val_losses.append(loss.item())
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

def compute_saliency(model, x):

    model.eval()

    if x.dim() == 3:
        x = x[0]

    x = x.clone().detach()
    x.requires_grad_(True)

    y = model(x.unsqueeze(0))  

    y = y.squeeze()

    model.zero_grad()
    y.backward()

    saliency = x.grad.abs().squeeze().cpu().numpy()

    return saliency, y.item()

In [61]:
input_length1, input_length2=X_POM_train.shape[1], X_POM_train.shape[2]
# =========================
# 1. Dataset
# =========================
class POMDataset(Dataset):
    def __init__(self, X, y):
        # (N, length) → (N, 1, length)
        X = X[:, None, :]
        
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 4)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# =========================
# 2. Lightning Model
# =========================
class POMModel(pl.LightningModule):
    def __init__(self, input_height, input_width, latent=8, lr=1e-3,):
        super().__init__()
        self.save_hyperparameters()

        self.model = nn.Sequential(
            # 1st Conv block
            nn.Conv2d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        
            # 2nd Conv block
            nn.Conv2d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        
            # 3rd Conv block
            nn.Conv2d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # channel compression
            nn.Conv2d(128, latent, kernel_size=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1,1)),
        
            nn.Flatten(),
            
            nn.ReLU(),
            nn.Dropout(0.1),
        
            nn.Linear(latent, 4)
        )

        self.loss_fn = nn.MSELoss()
        self.train_losses = []
        self.val_losses = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)

        self.train_losses.append(loss.item())
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)

        self.val_losses.append(loss.item())
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

# 3. Multimodal&multitask

In [ ]:
class Encoder1D(nn.Module):
    def __init__(self, input_length, latent):
        super().__init__()

        self.input_length = input_length
        self.latent = latent
        
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Conv1d(128, 16, kernel_size=1),
            nn.ReLU(),

            nn.Flatten(),

            nn.Linear(16 * (self.input_length // 2), self.latent)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Encoder2D(nn.Module):
    def __init__(self, latent):
        super().__init__()

        self.latent = latent
        
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        
            nn.Conv2d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        
            nn.Conv2d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(128, self.latent, kernel_size=1),
            nn.ReLU(),
            
            nn.AdaptiveAvgPool2d((1,1)),
        
            nn.Flatten()
        )
    
    def forward(self, x):
        return self.net(x)

In [ ]:
class MultiModalPG(pl.LightningModule):

    def __init__(self, input_length_IR=512, input_length_WAXS=512, latent=64, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        # =========================
        # encoders
        # =========================
        self.encoder_ir = Encoder1D(input_length_IR, latent)
        self.encoder_wax = Encoder1D(input_length_WAXS, latent)
        self.encoder_pom = Encoder2D(latent)

        self.fully_connected = nn.Sequential(
            nn.Linear(latent*3, latent),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(latent, 4)
        )
        
        # loss
        self.loss_fn = nn.MSELoss()
        self.train_loss_history = []
        self.val_loss_history = []

    # =========================
    # forward (projection space)
    # =========================
    def forward(self, x_ir, x_wax, x_pom):

        h_ir = self.encoder_ir(x_ir)
        h_wax = self.encoder_wax(x_wax)
        h_pom = self.encoder_pom(x_pom)

        h = torch.cat([h_ir, h_wax, h_pom], dim=1)

        out = self.fully_connected(h)

        return out
        
    def training_step(self, batch, batch_idx):
        x_ir, x_wax, x_pom, y = batch    
        y_pred = self(x_ir, x_wax, x_pom)
        loss = self.loss_fn(y_pred, y)

        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x_ir, x_wax, x_pom, y = batch    
        y_pred = self(x_ir, x_wax, x_pom)
        loss = self.loss_fn(y_pred, y)

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

    def on_train_epoch_end(self):
        if self.current_epoch%10==0:
            print(f"Epoch {self.current_epoch + 1}/{self.trainer.max_epochs}")
        metrics = self.trainer.callback_metrics
        if "train_loss" in metrics:
            self.train_loss_history.append(float(metrics["train_loss"].cpu()))

    def on_validation_epoch_end(self):
    
        metrics = self.trainer.callback_metrics
    
        if "val_loss" in metrics:
            self.val_loss_history.append(float(metrics["val_loss"].cpu()))

In [69]:
class AllDataset(Dataset):
    def __init__(self, X_IR, X_WAXS, X_POM, y):
        # (N, length) → (N, 1, length)
        X_IR = X_IR[:, None, :]
        X_WAXS = X_WAXS[:, None, :]
        X_POM = X_POM[:, None, :]
        
        self.X1 = torch.tensor(X_IR, dtype=torch.float32)
        self.X2 = torch.tensor(X_WAXS, dtype=torch.float32)
        self.X3 = torch.tensor(X_POM, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 4)

    def __len__(self):
        return len(self.X1)

    def __getitem__(self, idx):
        return self.X1[idx], self.X2[idx], self.X3[idx], self.y[idx]

In [ ]:
input_length_IR=X_IR_train.shape[1]
input_length_WAXS=X_WAXS_train.shape[1]

# The original never created this directory, so a 500-epoch run ended in a
# failing torch.save.
os.makedirs(path + "/model/Analysis2/" + model + "/", exist_ok=True)

for latent in latent_list:
    if not os.path.exists(path + "/model/Analysis2/"+model+"/Multi_"+model+"_"+str(latent)+"_"+str(seed)+".pth"):
        train_All_dataset = AllDataset(X_IR_train, X_WAXS_train, X_POM_train, y_train)
        test_All_dataset  = AllDataset(X_IR_test, X_WAXS_test, X_POM_test, y_test)
        
        train_All_loader = DataLoader(train_All_dataset, batch_size=16, shuffle=True, drop_last=True)
        test_All_loader  = DataLoader(test_All_dataset, batch_size=16, shuffle=False, drop_last=True)

        model_multi = MultiModalPG(input_length_IR=input_length_IR, input_length_WAXS=input_length_WAXS,
                                   latent=latent, lr=1e-4)
        
        trainer_multi = pl.Trainer(
            max_epochs=500,
            enable_progress_bar=True,
            accelerator="auto",
            logger=True
        )
        
        trainer_multi.fit(model_multi, train_All_loader, test_All_loader)
        
        ckpt = {
                "model_state": model_multi.state_dict(),
                "train_loss_history": model_multi.train_loss_history,
                "val_loss_history": model_multi.val_loss_history,
                "hparams": model_multi.hparams
            }
        torch.save(ckpt, path + "/model/Analysis2/"+model+"/Multi_"+model+"_"+str(latent)+"_"+str(seed)+".pth")

# 4. Evaluation of predicted values

In [139]:
prop_list=["Deg", "Strain", "Young", "CrysDeg"]   # was "CryDeg" (inconsistent with the data)

# The original never created this directory.
os.makedirs(path + "/result/Analysis2/" + model + "/", exist_ok=True)

for latent in latent_list:    
    model_IR2=IRModel(input_length=input_length_IR, latent=latent)
    model_IR2.load_state_dict(torch.load(path + "/model/Analysis2/PG1/IR_PG1_"+str(latent)+"_"+str(seed)+".pth", map_location="cpu",
                                         weights_only=False)["model_state"])
    model_WAXS2=WAXSModel(input_length=input_length_WAXS, latent=latent)
    model_WAXS2.load_state_dict(torch.load(path + "/model/Analysis2/PG1/WAXS_PG1_"+str(latent)+"_"+str(seed)+".pth", map_location="cpu",
                                         weights_only=False)["model_state"])
    model_POM2=POMModel(input_height=input_length1, input_width=input_length2, latent=latent)
    model_POM2.load_state_dict(torch.load(path + "/model/Analysis2/PG1/POM_PG1_"+str(latent)+"_"+str(seed)+".pth", map_location="cpu",
                                         weights_only=False)["model_state"])
    
    model_Mul2=MultiModalPG(input_length_IR=input_length_IR, input_length_WAXS=input_length_WAXS,
                            latent=latent, lr=1e-4)
    model_Mul2.load_state_dict(torch.load(path + "/model/Analysis2/"+model+"/Multi_"+model+"_"+str(latent)+"_"+str(seed)+".pth", map_location="cpu",
                                         weights_only=False)["model_state"])
    model_IR2.eval()
    model_WAXS2.eval()
    model_POM2.eval()
    model_Mul2.eval()
    
    train_All_dataset = AllDataset(X_IR_train, X_WAXS_train, X_POM_train, y_train)
    test_All_dataset  = AllDataset(X_IR_test, X_WAXS_test, X_POM_test, y_test)
    
    train_All_loader = DataLoader(train_All_dataset, batch_size=1, shuffle=False)
    test_All_loader  = DataLoader(test_All_dataset, batch_size=1, shuffle=False)

    y_IR_train, y_WAXS_train, y_POM_train, y_Mul_train, y_train_real = [], [], [], [], [] 
    y_IR_test,  y_WAXS_test,  y_POM_test,  y_Mul_test,  y_test_real  = [], [], [], [], [] 
    
    with torch.no_grad():
        for x_IR, x_WAXS, x_POM, y in train_All_loader:
            y_Mul_train.append(model_Mul2(x_IR, x_WAXS, x_POM).cpu().numpy()[0])
            y_train_real.append(y.cpu().numpy()[0])
            y_IR_train.append(model_IR2(x_IR).cpu().numpy()[0])
            y_WAXS_train.append(model_WAXS2(x_WAXS).cpu().numpy()[0])
            y_POM_train.append(model_POM2(x_POM).cpu().numpy()[0])

        for x_IR, x_WAXS, x_POM, y in test_All_loader:
            y_Mul_test.append(model_Mul2(x_IR, x_WAXS, x_POM).cpu().numpy()[0])
            y_test_real.append(y.cpu().numpy()[0])
            y_IR_test.append(model_IR2(x_IR).cpu().numpy()[0])
            y_WAXS_test.append(model_WAXS2(x_WAXS).cpu().numpy()[0])
            y_POM_test.append(model_POM2(x_POM).cpu().numpy()[0])
    
    df_train = pd.concat([pd.DataFrame(y_IR_train, columns=["IR,"+i for i in prop_list]),
                         pd.DataFrame(y_WAXS_train, columns=["WAXS,"+i for i in prop_list]),
                         pd.DataFrame(y_POM_train, columns=["POM,"+i for i in prop_list]),
                        pd.DataFrame(y_Mul_train, columns=["Mul,"+i for i in prop_list]),
                         pd.DataFrame(y_train_real, columns=["Real,"+i for i in prop_list])],
                        axis=1)
    df_test  = pd.concat([pd.DataFrame(y_IR_test, columns=["IR,"+i for i in prop_list]),
                         pd.DataFrame(y_WAXS_test, columns=["WAXS,"+i for i in prop_list]),
                         pd.DataFrame(y_POM_test, columns=["POM,"+i for i in prop_list]),
                        pd.DataFrame(y_Mul_test, columns=["Mul,"+i for i in prop_list]),
                         pd.DataFrame(y_test_real, columns=["Real,"+i for i in prop_list])],
                        axis=1)

    df_train.to_csv(path+'/result/Analysis2/'+model+'/y_train_'+model+'_'+str(latent)+"_"+str(seed)+'.csv')
    df_test.to_csv(path+'/result/Analysis2/'+model+'/y_test_'+model+'_'+str(latent)+"_"+str(seed)+'.csv')

# Reproducibility notes (added)

This notebook is the *cleaned* copy of `Fig4_Multimodal_multitask_original.ipynb`.
Only the points below were changed; the model definitions and the training loop are
untouched.

1. The hard-coded absolute path
   `path = "/Users/amamotoyoshifumi/Desktop/Contrastive/analysis"` was removed. It
   overrode `path = "./"`, so the notebook could not run anywhere else.
2. `seed = int(args[1])` (with `args = sys.argv`) was replaced by a direct
   assignment. The original raised `IndexError` when run from Jupyter.
3. `model = "PG3"` is kept, but the label is documented: it names the *multimodal*
   checkpoints only, while the single-modality checkpoints loaded in section 4 are
   hard-coded to `PG1`. If the two must agree, change one of them.
4. `POMModel` logged without a value
   (`self.log("train_loss", on_step=False, on_epoch=True, prog_bar=True)`), which
   is a runtime error; `loss` was added to both `training_step` and
   `validation_step`. `IRModel` and `WAXSModel` were already correct.
5. `model/Analysis2/<model>/` and `result/Analysis2/<model>/` are now created with
   `os.makedirs`. Neither existed in the original, so a 500-epoch run was lost on
   the final `torch.save` / `to_csv`.
6. The column label `"CryDeg"` was corrected to `"CrysDeg"` so that the output CSV
   matches the target name used everywhere else.

Not changed (deliberate): the architectures, the group-wise split, the four target
definitions, the epoch counts, and the loss.

Points that still need attention (see the review document):

- Training is not fully reproducible: `seed` fixes the split, but
  `pytorch_lightning.seed_everything(seed)` is never called and
  `trainer(deterministic=True)` is not set.
- `drop_last=True` is used on the **test** loader as well, so up to 15 of the 153
  samples are silently dropped from the validation loss.
- The multimodal model is trained on row-normalised inputs, while the PG1
  single-modality checkpoints were produced by a different preprocessing chain
  (see the review). Verify that they are consistent, or retrain.
- No best-checkpoint selection and no early stopping; the last epoch is used.
- The output CSVs are in standardised space. De-standardise with the **training**
  statistics before reporting R^2.
- `X_IR`/`X_WAXS` must have even length and the POM images must be a multiple of 8
  (`input_length // 2` and three successive `MaxPool2d`).
